# Part 1: SRE Agent Foundations
## Setting up your SRE Multi-Agent System

## Overview

In this first notebook, you'll set up the foundation for building a production-ready Site Reliability Engineering (SRE) Agent. We'll establish your development environment, understand the multi-agent architecture, and get your first SRE investigation running locally.

By the end of this notebook, you'll have a working local SRE agent that can investigate infrastructure issues using synthetic demo data.

### Tutorial Details

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Foundation Setup                                                                 |
| Agent type          | Multi-Agent System                                                               |
| Agentic Framework   | LangGraph with Strands Agents                                                   |
| LLM models          | Amazon Nova Pro, Anthropic Claude Sonnet 3.7                                   |
| Tutorial components | Environment setup, Architecture overview, Demo backend, Local agent testing     |
| Tutorial vertical   | DevOps/SRE                                                                       |
| Example complexity  | Intermediate                                                                     |
| Duration           | 30-45 minutes                                                                    |

### Learning Objectives

- **Understand SRE Agent Architecture**: Learn how multi-agent systems work together
- **Set Up Development Environment**: Configure Python, dependencies, and AWS credentials
- **Configure Demo Backend**: Start mock services that simulate real infrastructure
- **Run First Investigation**: Execute your first SRE investigation locally
- **Validate Setup**: Ensure all components are working correctly

### Architecture Overview

The SRE Agent is a sophisticated multi-agent system built on **LangGraph** that orchestrates specialized agents:

```
┌─────────────────────────────────────────────────────────────────┐
│                    SRE Agent System                             │
│                                                                 │
│  ┌──────────────┐    ┌─────────────────────────────────────┐   │
│  │  Supervisor  │───▶│         Specialist Agents           │   │
│  │    Agent     │    │  ┌─────┬─────┬─────┬──────────────┐ │   │
│  │              │    │  │ K8s │Logs │Metr-│   Runbooks   │ │   │
│  │ • Planning   │    │  │Agent│Agent│ics  │    Agent     │ │   │
│  │ • Routing    │    │  │     │     │Agent│              │ │   │
│  │ • Memory     │    │  └─────┴─────┴─────┴──────────────┘ │   │
│  └──────────────┘    └─────────────────────────────────────┘   │
│           │                           │                       │
└───────────┼───────────────────────────┼───────────────────────┘
            │                           │
            ▼                           ▼
    ┌──────────────┐            ┌─────────────────┐
    │   Memory     │            │  Demo Backend   │
    │  (Future)    │            │   Services      │
    └──────────────┘            │                 │
                                │ • K8s API       │
                                │ • Logs API      │
                                │ • Metrics API   │
                                │ • Runbooks API  │
                                └─────────────────┘
```

## Prerequisites Check

Let's start by validating that your environment meets the requirements for this workshop.

In [ ]:
# Install workshop dependencies
!pip install --quiet --upgrade pip
!pip install --quiet -r ../requirements.txt

In [ ]:
# Import required libraries and workshop utilities
import sys
import os
from pathlib import Path

# Add workshop helpers to path
sys.path.append(str(Path().absolute().parent / "helpers"))

# Add main SRE agent code to path
sys.path.append(str(Path().absolute().parent.parent.parent))

from workshop_utils import load_workshop_config, get_aws_account_id, get_private_ip
from validation_helpers import WorkshopValidator, print_validation_summary
from sre_scenarios import SREScenarios, get_scenario_prompts

print("✓ Workshop utilities imported successfully")

In [ ]:
# Load workshop configuration
config = load_workshop_config()
validator = WorkshopValidator(config)

print("Workshop Configuration:")
print(f"  AWS Region: {config.get('aws', {}).get('region', 'us-east-1')}")
print(f"  Workshop Prefix: {config.get('names', {}).get('prefix', 'sre-workshop')}")
print("\nValidating environment setup...")

# Validate environment
env_results = validator.validate_environment_setup()
print_validation_summary({"Environment Setup": env_results})

### ⚠️ Important: Environment Variables Setup

Before continuing, you need to set up your environment variables. Create a `.env` file in the SRE agent root directory:

In [ ]:
# Check if .env file exists, create template if needed
env_file = Path().absolute().parent.parent.parent / "sre_agent" / ".env"
env_example = Path().absolute().parent.parent.parent / "sre_agent" / ".env.example"

if not env_file.exists() and env_example.exists():
    print("Creating .env file from template...")
    import shutil
    shutil.copy(str(env_example), str(env_file))
    print(f"✓ .env file created at {env_file}")
    print("\n⚠️  IMPORTANT: Edit the .env file and add your API keys:")
    print(f"   - ANTHROPIC_API_KEY=your_key_here")
    print(f"   - LLM_PROVIDER=bedrock (or anthropic)")
    print(f"   - USER_ID=Alice")
elif env_file.exists():
    print(f"✓ .env file exists at {env_file}")
else:
    print(f"❌ Could not find .env.example file. Please create .env manually.")

# Load environment variables
from dotenv import load_dotenv
load_dotenv(env_file)
print("\n✓ Environment variables loaded")

## Understanding the SRE Agent Architecture

Before we start building, let's understand the key architectural components:

In [ ]:
# Explore the SRE agent structure
sre_root = Path().absolute().parent.parent.parent

print("SRE Agent Project Structure:")
print("─" * 50)

# Key directories to highlight
key_dirs = {
    "sre_agent/": "Core multi-agent system code",
    "sre_agent/config/": "Agent configuration and prompts", 
    "sre_agent/memory/": "Memory integration (used in Part 4)",
    "backend/": "Demo backend services (K8s, Logs, Metrics, Runbooks)",
    "gateway/": "AgentCore Gateway configuration (used in Part 2)",
    "deployment/": "Production deployment scripts (used in Part 5)"
}

for dir_path, description in key_dirs.items():
    full_path = sre_root / dir_path
    exists = "✓" if full_path.exists() else "✗"
    print(f"{exists} {dir_path:<20} {description}")

print("\n🔍 Multi-Agent Components:")
print("─" * 30)

agent_files = {
    "supervisor.py": "Central coordinator with memory access",
    "agent_nodes.py": "Specialist agent implementations", 
    "multi_agent_langgraph.py": "Main LangGraph orchestration",
    "graph_builder.py": "Graph construction logic"
}

for file_name, description in agent_files.items():
    file_path = sre_root / "sre_agent" / file_name
    exists = "✓" if file_path.exists() else "✗"
    print(f"{exists} {file_name:<25} {description}")

### Agent Specialization

Each specialist agent has specific tools and expertise:

In [ ]:
# Load and display agent configuration
import yaml

agent_config_path = sre_root / "sre_agent" / "config" / "agent_config.yaml"

if agent_config_path.exists():
    with open(agent_config_path, 'r') as f:
        agent_config = yaml.safe_load(f)
    
    print("Agent Specializations and Tools:")
    print("═" * 60)
    
    for agent_name, config in agent_config.get('agents', {}).items():
        print(f"\n🤖 {config['name']}")
        print(f"   Description: {config['description']}")
        print(f"   Tools: {', '.join(config['tools'])}")
        
    print(f"\n🌐 Global Tools: {', '.join(agent_config.get('global_tools', []))}")
    
else:
    print("❌ Agent configuration not found. Please check your setup.")

## Setting Up Demo Backend Services

The SRE agent needs backend services to investigate. We'll start the demo backend that simulates real infrastructure APIs with synthetic data.

In [ ]:
# Get EC2 private IP for backend binding (or localhost for local development)
private_ip = get_private_ip()
print(f"Backend will bind to: {private_ip}")

# Check if SSL certificates are available for HTTPS
ssl_cert_path = "/opt/ssl/fullchain.pem"
ssl_key_path = "/opt/ssl/privkey.pem"

use_ssl = os.path.exists(ssl_cert_path) and os.path.exists(ssl_key_path)
protocol = "https" if use_ssl else "http"

print(f"SSL certificates available: {use_ssl}")
print(f"Backend protocol: {protocol.upper()}")

if use_ssl:
    print("⚠️  Using SSL certificates - required for AgentCore Gateway integration")
else:
    print("⚠️  No SSL certificates found - will use HTTP (local development only)")

In [ ]:
# Start demo backend services
import subprocess
import time
from workshop_utils import run_backend_servers

print("Starting demo backend services...")
print("This may take 30-60 seconds to fully initialize.")

try:
    # Start backend servers
    if use_ssl:
        backend_process = run_backend_servers(
            host=private_ip,
            ssl_cert=ssl_cert_path,
            ssl_key=ssl_key_path
        )
    else:
        backend_process = run_backend_servers(host=private_ip)
    
    # Wait for services to start
    print("⏳ Waiting for services to start...")
    time.sleep(15)
    
    # Define backend service URLs
    base_ports = [8011, 8012, 8013, 8014]  # K8s, Logs, Metrics, Runbooks
    service_names = ["Kubernetes", "Logs", "Metrics", "Runbooks"]
    
    backend_urls = []
    for port in base_ports:
        backend_urls.append(f"{protocol}://{private_ip}:{port}")
    
    print("\n✓ Backend services started:")
    for service, url in zip(service_names, backend_urls):
        print(f"  {service:<12} {url}")
        
    # Store for later use
    globals()['backend_process'] = backend_process
    globals()['backend_urls'] = backend_urls
    
except Exception as e:
    print(f"❌ Failed to start backend services: {e}")
    print("Please check the backend setup and try again.")

In [ ]:
# Validate backend services are responding
import requests
import time

print("Validating backend services...")
print("This may take a moment as services finish initialization.")

# Give services more time to fully start
time.sleep(10)

validation_results = {}
for service, url in zip(["Kubernetes", "Logs", "Metrics", "Runbooks"], backend_urls):
    try:
        # Use longer timeout for initial startup
        response = requests.get(f"{url}/health", timeout=10, verify=False)
        is_healthy = response.status_code == 200
        validation_results[f"{service.lower()}_service"] = is_healthy
        
        status = "✓ Healthy" if is_healthy else f"✗ Unhealthy ({response.status_code})"
        print(f"  {service:<12} {status}")
        
    except Exception as e:
        validation_results[f"{service.lower()}_service"] = False
        print(f"  {service:<12} ✗ Failed: {str(e)[:50]}...")

# Summary
healthy_services = sum(validation_results.values())
total_services = len(validation_results)

print(f"\nBackend Services Status: {healthy_services}/{total_services} healthy")

if healthy_services == total_services:
    print("🎉 All backend services are running successfully!")
elif healthy_services > 0:
    print("⚠️  Some services are running. You can continue but may see limited functionality.")
else:
    print("❌ No services are responding. Please check the backend setup.")

## Your First SRE Investigation

Now let's run your first SRE investigation! We'll start with a simple scenario to test the local setup.

In [ ]:
# Explore available test scenarios
scenarios = SREScenarios.get_all_scenarios()
scenario_prompts = get_scenario_prompts()

print("Available SRE Investigation Scenarios:")
print("═" * 70)

for i, scenario in enumerate(scenarios[:4], 1):  # Show first 4 for brevity
    print(f"\n{i}. {scenario.title}")
    print(f"   Severity: {scenario.severity.value.upper()}")
    print(f"   Difficulty: {scenario.difficulty.title()}")
    print(f"   Time: {scenario.estimated_time}")
    print(f"   Services: {', '.join(scenario.affected_services)}")
    print(f"   Description: {scenario.description[:100]}...")

print(f"\n... and {len(scenarios)-4} more scenarios available for testing.")

# Select first scenario for demo
demo_scenario = scenarios[1]  # Pod crash loop - good starter scenario
demo_prompt = scenario_prompts[demo_scenario.id]

print(f"\n🎯 We'll start with: {demo_scenario.title}")
print(f"Prompt: {demo_prompt}")

In [ ]:
# Configure the SRE agent for local testing
import logging

# Set up logging to see agent activity
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(name)s | %(message)s')

# Import SRE agent components
try:
    from sre_agent.multi_agent_langgraph import SREAgent
    from sre_agent.config import SREConstants
    print("✓ SRE agent components imported successfully")
except ImportError as e:
    print(f"❌ Failed to import SRE agent: {e}")
    print("Please check your Python path and SRE agent installation")

In [ ]:
# For local testing, we'll simulate the investigation without requiring gateway setup
# This gives you a preview of what the full system will do

print("🔍 Simulating SRE Investigation Process...")
print("═" * 60)

print(f"\n📋 Scenario: {demo_scenario.title}")
print(f"   User Input: {demo_prompt}")
print(f"   Affected Services: {', '.join(demo_scenario.affected_services)}")
print(f"   Severity: {demo_scenario.severity.value.upper()}")

print("\n🤖 Multi-Agent Investigation Plan:")
print("   1. Supervisor analyzes the problem and creates investigation plan")
print("   2. Routes to specialist agents based on symptoms:")
for tool in demo_scenario.expected_tools_used:
    if 'pod' in tool or 'cluster' in tool or 'deployment' in tool:
        agent_type = "Kubernetes Agent"
    elif 'log' in tool:
        agent_type = "Logs Agent" 
    elif 'metric' in tool or 'resource' in tool:
        agent_type = "Metrics Agent"
    else:
        agent_type = "Runbooks Agent"
    print(f"      → {agent_type}: {tool}")

print("   3. Agents execute tools in parallel and return findings")
print("   4. Supervisor aggregates results and provides recommendations")

print("\n📊 Expected Investigation Steps:")
for i, step in enumerate(demo_scenario.expected_investigation_steps, 1):
    print(f"   {i}. {step}")

print("\n🎯 Learning Objectives for This Scenario:")
for obj in demo_scenario.learning_objectives:
    print(f"   • {obj}")

print("\n⚡ In the next notebooks, this investigation will run automatically with real agent responses!")

## Workshop Progress Validation

Let's validate that everything is set up correctly for the next parts of the workshop:

In [ ]:
# Run comprehensive validation
print("Running comprehensive workshop validation...")
print("This checks all components needed for upcoming notebook sections.")

all_validation_results = {}

# 1. Environment setup (already done)
all_validation_results["Environment"] = validator.validate_environment_setup()

# 2. Backend services  
all_validation_results["Backend Services"] = validation_results  # From earlier check

# 3. Basic agent functionality check
print("\nValidating SRE agent can be imported and configured...")
try:
    # Test basic imports and configuration loading
    from sre_agent.constants import SREConstants
    from sre_agent.prompt_loader import prompt_loader
    from sre_agent.llm_utils import create_llm_with_error_handling
    
    # Test configuration loading
    constants = SREConstants()
    
    # Test LLM creation (without actually calling)
    provider = os.getenv('LLM_PROVIDER', 'bedrock')
    
    agent_validation = {
        'imports_successful': True,
        'constants_loaded': True,
        'llm_provider_configured': bool(provider),
        'ready_for_gateway_integration': True
    }
    
    print("✓ SRE agent components validated")
    
except Exception as e:
    agent_validation = {
        'imports_successful': False,
        'constants_loaded': False, 
        'llm_provider_configured': False,
        'ready_for_gateway_integration': False
    }
    print(f"✗ SRE agent validation failed: {e}")

all_validation_results["SRE Agent"] = agent_validation

# Print comprehensive summary
print_validation_summary(all_validation_results)

## Understanding What You've Built

Congratulations! You now have the foundation of a sophisticated SRE agent system. Let's review what you've accomplished:

In [ ]:
print("🎉 Foundation Setup Complete!")
print("═" * 50)

print("\n✅ What You've Accomplished:")
print("   • Set up Python environment with all required dependencies")
print("   • Configured AWS credentials and validated permissions")
print("   • Started demo backend services (K8s, Logs, Metrics, Runbooks APIs)")
print("   • Understood the multi-agent architecture and specialization")
print("   • Validated all components for upcoming workshop sections")

print("\n🏗️ Architecture Components Ready:")
print("   • Supervisor Agent: Central coordinator with planning capability")
print("   • Specialist Agents: 4 domain-specific agents ready for integration")
print("   • Demo Backend: 4 API services with realistic synthetic data")
print("   • LangGraph Framework: Multi-agent orchestration system")

print("\n🚀 Ready for Next Steps:")
print("   📖 Notebook 2: AgentCore Gateway Integration")
print("      - Create secure MCP tools from backend APIs")
      - Set up authentication with Amazon Cognito")
print("      - Test individual tool invocations")

print("   📖 Notebook 3: Multi-Agent System")
print("      - Build complete LangGraph agent orchestration")
print("      - Test realistic SRE investigations")
print("      - Validate agent collaboration")

print("\n💡 Key Learning Points:")
print("   • Multi-agent systems can specialize in different domains")
print("   • Synthetic data enables safe testing without production impact")
print("   • Proper environment setup is critical for complex AI systems")
print("   • AWS services can be mocked for development and learning")

# Show current resource usage
healthy_services = sum(validation_results.values())
print(f"\n📊 Current Status:")
print(f"   Demo Services Running: {healthy_services}/4")
print(f"   Backend Protocol: {protocol.upper()}")
print(f"   Ready for Gateway Integration: {'Yes' if healthy_services >= 3 else 'Partial'}")

## Cleanup (Optional)

If you want to stop the demo backend services (you can restart them later), run the cell below:

In [ ]:
# Optional: Stop backend services
# Uncomment and run if you want to clean up

# if 'backend_process' in globals():
#     print("Stopping demo backend services...")
#     backend_process.terminate()
#     backend_process.wait()
#     print("✓ Backend services stopped")
# else:
#     print("No backend process found to stop")

print("💡 Backend services are still running for use in the next notebook.")
print("   You can proceed directly to Notebook 2: Gateway and MCP Tools")

## Next Steps

🎯 **You're ready for the next part of the workshop!**

**Continue to:** [Notebook 2: Gateway and MCP Tools](02-gateway-and-mcp-tools.ipynb)

In the next notebook, you'll:
- Create an AgentCore Gateway to secure your backend APIs
- Set up Amazon Cognito for authentication
- Transform your backend services into MCP tools
- Test agent-gateway communication

---

### 💭 Reflection Questions
- How does the multi-agent approach differ from a single large agent?
- What are the benefits of having specialized agents for different domains?
- How might this architecture scale in a real production environment?

### 🔗 Additional Resources
- [Amazon Bedrock AgentCore Documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/agents.html)
- [LangGraph Multi-Agent Patterns](https://langchain-ai.github.io/langgraph/)
- [SRE Best Practices](https://sre.google/books/)